In [1]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U transformers peft trl datasets bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.5 MB/s eta 0:00:00


### Listing 7.7: Setting Up the GRPO Environment and Dependencies

In [ ]:
import re
import os
import sys
import glob
import torch
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig
from trl import GRPOTrainer, GRPOConfig

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

print(f"CUDA available: {torch.cuda.is_available()} | Compute Dtype: {compute_dtype}")

### Listing 7.8: Defining System Prompt and Formatting GSM8K Training Dataset

In [ ]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\n"
    "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\n"
    "The reasoning process and answer are enclosed within tags. The answer must be a single integer.\n"
    "Example:\n"
    "<reasoning>\n"
    "We know that 2 + 2 = 4.\n"
    "</reasoning>\n"
    "<answer>4</answer>"
)

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

dataset = load_dataset("openai/gsm8k", "main", split="train")
dataset = dataset.shuffle(seed=42)

def format_gsm8k(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
        ],
        "answer": extract_hash_answer(example["answer"]),
    }

gsm8k_train = dataset.map(format_gsm8k)
print("Sample Prompt Structure Preview:\n", gsm8k_train[0]["prompt"])

### Listing 7.9: Implementing the Graduated Format Reward Function

In [ ]:
# 1. Graduated Format Reward (Max: 0.5)
def format_reward(completions, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    rewards = []
    for response in responses:
        score = 0.0
        if "<reasoning>" in response: score += 0.1
        if "</reasoning>" in response: score += 0.1
        if "<answer>" in response: score += 0.1
        if "</answer>" in response: score += 0.1

        pattern = r"^<reasoning>[\s\S]*?<\/reasoning>\s*<answer>[\s\S]*?<\/answer>$"
        if re.match(pattern, response):
            score += 0.1

        rewards.append(score)
    return rewards

### Listing 7.10: Implementing the Math Step / Equation Reward Function

In [ ]:
# 2. Math Step / Equation Reward (Replaces raw length reward, Max: 0.5)
def math_step_reward(completions, **kwargs):
    """
    Rewards generating actual mathematical calculations (e.g., '16 - 7 = 9')
    rather than just filling space with words.
    """
    responses = [completion[0]["content"] for completion in completions]
    rewards = []
    for response in responses:
        match = re.search(r"<reasoning>(.*?)</reasoning>", response, re.DOTALL)
        if match:
            text = match.group(1)
            # Find explicit arithmetic operations like "16 - 7 = 9" or "180 / 3 = 60"
            equations = re.findall(r"\d+\s*[\+\-\*/]\s*\d+\s*=\s*\d+", text)
            eq_count = len(equations)

            # Award +0.15 per equation, capped at 0.5 max
            rewards.append(min(0.5, eq_count * 0.15))
        else:
            rewards.append(0.0)
    return rewards

### Listing 7.11: Implementing the Partial Correctness Reward Function

In [ ]:
# 3. Partial Correctness Reward (Max: 0.5)
def partial_correctness_reward(completions, answer, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    rewards = []
    for response, ans in zip(responses, answer):
        if ans and re.search(r"\b" + re.escape(str(ans)) + r"\b", response):
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

### Listing 7.12: Implementing the Strict Final Correctness Reward Function

In [ ]:
def extract_last_number(text, start_tag="<answer>", end_tag="</answer>"):
    pattern = re.escape(start_tag) + r"(.*?)" + re.escape(end_tag)
    matches = re.findall(pattern, text, re.DOTALL)
    text_to_search = matches[-1] if matches else text
    numbers = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text_to_search)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return ""

def extract_answer(text):
    return extract_last_number(text)

# 4. Strict Final Correctness Reward (Max: 2.0)
def correctness_reward(completions, answer, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    extracted = [extract_last_number(response) for response in responses]
    rewards = [2.0 if ext == ans else 0.0 for ext, ans in zip(extracted, answer)]
    return rewards

### Listing 7.13: Loading Base Model and Initializing Evaluation Variables

In [ ]:
HF_REPO_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_NAME = HF_REPO_ID.split("/")[-1].lower()
tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID)

base_model_eval = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID, dtype=compute_dtype, device_map="auto"
)
base_model_eval.eval()
base_model_eval.generation_config.max_length = None

eval_dataset_100 = load_dataset("openai/gsm8k", "main", split="test").select(range(100))
pre_grpo_responses_100 = []
pre_grpo_correct = 0
pre_grpo_format = 0

pattern = r"^<reasoning>[\s\S]*?<\/reasoning>\s*<answer>[\s\S]*?<\/answer>$"

### Listing 7.14: Running the Pre-GRPO Benchmark Evaluation

In [ ]:
print("=== Running Pre-GRPO Benchmark on 100 GSM8K Test Problems ===")

for i, example in enumerate(eval_dataset_100):
    question = example["question"]
    ground_truth = extract_hash_answer(example["answer"])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    input_ids = inputs if isinstance(inputs, torch.Tensor) else inputs["input_ids"]
    input_ids = input_ids.to("cuda")

    with torch.no_grad():
        outputs = base_model_eval.generate(
            input_ids=input_ids, max_new_tokens=2048, pad_token_id=tokenizer.eos_token_id
        )

    resp = tokenizer.decode(
        outputs[0][input_ids.shape[1] :], skip_special_tokens=True
    ).strip()

    pre_grpo_responses_100.append(resp)
    ext_ans = extract_last_number(resp)

    if ext_ans == ground_truth:
        pre_grpo_correct += 1

    if re.match(pattern, resp):
        pre_grpo_format += 1

    if (i + 1) % 20 == 0:
        print(
            f"Evaluated {i + 1}/100 problems | Current Accuracy: {pre_grpo_correct / (i + 1) * 100:.1f}%"
        )

pre_acc = (pre_grpo_correct / 100) * 100
pre_fmt = (pre_grpo_format / 100) * 100

print(f"\n>>> PRE-GRPO BENCHMARK (100 Problems) <<<")
print(f"Format Compliance: {pre_fmt:.1f}% ({pre_grpo_format}/100)")
print(f"Math Accuracy:     {pre_acc:.1f}% ({pre_grpo_correct}/100)\n")

del base_model_eval
torch.cuda.empty_cache()

### Listing 7.15: Configuring LoRA, GRPO Parameters, and Launching GRPOTrainer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID)

peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.0,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj", "up_proj", "down_proj",
    ],
)

training_args = GRPOConfig(
    use_vllm=True,
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=25,
    beta=0.02,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    gradient_accumulation_steps=8,
    per_device_train_batch_size=2,
    num_generations=16,
    temperature=1.0,
    max_completion_length=2048,
    max_steps=850,
    logging_steps=50,
    save_steps=50,
    max_grad_norm=1.0,
    report_to="none",
    output_dir= MODEL_NAME + "-grpo-output",
)

model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID, dtype=compute_dtype, device_map="auto"
)

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}


trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward,
        math_step_reward,
        partial_correctness_reward,
        correctness_reward,
        ],
    args=training_args,
    train_dataset=gsm8k_train,
    peft_config=peft_config,
)

trainer.train()

### Listing 7.16: Merging LoRA Adapter and Saving the Reasoning Model

In [ ]:
merged_model = trainer.model.merge_and_unload()
tokenizer.save_pretrained(MODEL_NAME + "-grpo-adapter")
merged_model.save_pretrained(MODEL_NAME + "-grpo-adapter")

### Listing 7.17: Running the Post-GRPO Evaluation Benchmark

In [ ]:
merged_model.eval()
merged_model.generation_config.max_length = None

post_grpo_responses_100 = []
post_grpo_correct = 0
post_grpo_format = 0

print("=== Running Post-GRPO Benchmark on 100 GSM8K Test Problems ===")

for i, example in enumerate(eval_dataset_100):
    question = example["question"]
    ground_truth = extract_hash_answer(example["answer"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )

    input_ids = inputs if isinstance(inputs, torch.Tensor) else inputs["input_ids"]
    input_ids = input_ids.to("cuda")

    with torch.no_grad():
        outputs = merged_model.generate(
            input_ids=input_ids, max_new_tokens=2048,
            pad_token_id=tokenizer.eos_token_id)

    post_resp = tokenizer.decode(
        outputs[0][input_ids.shape[1] :], skip_special_tokens=True
    ).strip()

    post_grpo_responses_100.append(post_resp)
    ext_ans = extract_last_number(post_resp)

    if ext_ans == ground_truth:
        post_grpo_correct += 1

    if re.match(pattern, post_resp):
        post_grpo_format += 1

    if (i + 1) % 20 == 0:
        print(f"Evaluated {i + 1}/100 problems | Current Accuracy: "
              f"{post_grpo_correct / (i + 1) * 100:.1f}%")

post_acc = (post_grpo_correct / 100) * 100
post_fmt = (post_grpo_format / 100) * 100
acc_delta = post_acc - pre_acc
fmt_delta = post_fmt - pre_fmt

print("\n" + "=" * 60)
print(">>> FINAL GSM8K 100-PROBLEM BENCHMARK RESULTS <<<")
print(f"Pre-GRPO Format Compliance:  {pre_fmt:.1f}%")
print(f"Post-GRPO Format Compliance: {post_fmt:.1f}%")
print(f"Format Compliance Delta:    {'+' if fmt_delta >= 0 else ''}{fmt_delta:.1f}%\n")
print(f"Pre-GRPO Math Accuracy:      {pre_acc:.1f}%")
print(f"Post-GRPO Math Accuracy:     {post_acc:.1f}%")
print(f"Math Accuracy Delta:        {'+' if acc_delta >= 0 else ''}{acc_delta:.1f}%")
print("=" * 60 + "\n")
print("=== Detailed 5-Example Side-by-Side Comparison ===")

### Listing 7.18: Displaying Side-by-Side Reasoning Comparison

In [ ]:
for i in range(5):
    example = eval_dataset_100[i]
    question = example["question"]
    ground_truth = extract_hash_answer(example["answer"])
    print(f"=== Test Example {i + 1} ===")
    print(f"Question: {question}")
    print(f"Ground Truth Answer: {ground_truth}")
    print("Before GRPO (Pre-trained Model):"
          f"\n{pre_grpo_responses_100[i]}\n")
    print("After GRPO (RL Aligned Model):"
          f"\n{post_grpo_responses_100[i]}\n")
    print("-" * 80 + "\n")